In [ ]:
# -*- coding: utf-8 -*-
"""[Stage 2] Pipeline Generator Dataset DPO FASILKOM.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/16dBydk_H8bpsJCYE9sYRHrxYTYg61sD8
"""

# -*- coding: utf-8 -*-
"""
stage_2_dpo_dataset_generator_fasilkom.py
=============================================================================
Pipeline Generator Dataset DPO — FASILKOM
Triple-Dimension Semantic-Lexical Proxy Scoring (sesuai metodologi paper)
=============================================================================

ARSITEKTUR PIPELINE
-------------------
Untuk setiap pertanyaan (175 soal akademik dari CSV):

  1. Retrieve top-2 halaman dokumen via ColPali v1.3-hf + Qdrant.

  2. Generate 3 kandidat jawaban menggunakan Qwen2-7B-Instruct:
       - chosen_conservative  : WITH context, temperature=0.4
       - chosen_creative      : WITH context, temperature=1.2
       - rejected             : WITHOUT context, temperature=1.8
     (System prompt berbeda sesuai paper, lihat Cell 13)

  3. Scoring setiap kandidat dengan S_final (Triple-Dimension Proxy):
       S_final = 0.4 x Sem_grounding + 0.4 x Sem_relevance + 0.2 x Lexical

       - Sem_grounding : cosine(e_response, e_context)   -- anti-halusinasi
       - Sem_relevance : cosine(e_response, e_query)     -- on-topic
       - Lexical       : |T_resp n T_ctx| / |T_resp|    -- keyword overlap

     Embedding: paraphrase-multilingual-MiniLM-L12-v2 (CPU, mendukung B.Indonesia)
     SEMUA embedding (response, context, query) dihitung dalam 1 forward pass.

  4. Pair selection dengan margin threshold >= 0.15:
       chosen   = kandidat WITH-context skor tertinggi
       rejected = kandidat WITHOUT-context
       Jika margin < 0.15 -> baris DILEWATI

  5. Output CSV triplet DPO:
       kode_pertanyaan | topik | pertanyaan | sumber_file |
       context_retrieved | sources_summary |
       chosen | score_chosen | temp_chosen | label_chosen |
       sem_grounding_chosen | sem_relevance_chosen | lexical_chosen |
       rejected | score_rejected | temp_rejected |
       sem_grounding_rejected | sem_relevance_rejected | lexical_rejected |
       score_margin | timestamp

REFERENSI DESAIN
----------------
- Scoring terinspirasi framework KoMeP (Korean Medical Preference).
- Bobot (0.4, 0.4, 0.2) dikalibrasi via grid-search pada held-out 50 query.
- Threshold margin 0.15 sesuai pendekatan kualitas dataset legal Q&A.
- Contrastive sampling terinspirasi prinsip MAGNET.
- Stop-words (60 kata) + filter token < 4 karakter sesuai paper.

CATATAN TEKNIS
--------------
- Retriever : vidore/colpali-v1.3-hf  (ColPaliForRetrieval, bfloat16, GPU)
- Generator : Qwen/Qwen2-7B-Instruct  (4-bit BnB, GPU)
- Scorer    : sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 (CPU)
- TOP_K     : 3
- Checkpoint otomatis setiap CHECKPOINT_EVERY baris.
- Resume otomatis jika sesi Colab putus.
=============================================================================
"""

# ============================================================
# [Cell 0] Install dependency
# ============================================================
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# Uncomment dan jalankan sekali di Colab:
!pip install -q --no-cache-dir \
     "transformers>=4.49.0" \
     qdrant-client \
     bitsandbytes \
     accelerate \
     pymupdf \
     pytesseract \
     pillow \
     psutil \
     pandas \
     "sentence-transformers>=2.7.0"
!apt-get install -q -y tesseract-ocr tesseract-ocr-ind

# ============================================================
# [Cell 1] Imports & Konfigurasi
# ============================================================
# Suppress FutureWarning dari bitsandbytes internal
# (bitsandbytes/backends/cuda/ops.py: _check_is_size deprecation)
# Warning ini bukan dari kode kita dan tidak mempengaruhi fungsionalitas.
import warnings
warnings.filterwarnings(
    "ignore",
    message=".*_check_is_size.*",
    category=FutureWarning,
    module="bitsandbytes",
)

import re
import gc
import json
import time
import logging
import datetime
import traceback

import torch
import psutil
import fitz          # PyMuPDF
import numpy as np
import pandas as pd
from PIL import Image
from sentence_transformers import SentenceTransformer

from qdrant_client import QdrantClient
from transformers import (
    ColPaliForRetrieval, ColPaliProcessor,
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
)
from accelerate.utils import release_memory

# ---- Google Colab secrets -----------------------------------
try:
    from google.colab import drive, userdata
    drive.mount("/content/drive", force_remount=False)
    HF_TOKEN       = userdata.get("HF_TOKEN")
    QDRANT_URL     = userdata.get("QDRANT_URL")
    QDRANT_API_KEY = userdata.get("QDRANT_API_KEY")
except Exception:
    HF_TOKEN       = os.environ.get("HF_TOKEN", "")
    QDRANT_URL     = os.environ.get("QDRANT_URL", "")
    QDRANT_API_KEY = os.environ.get("QDRANT_API_KEY", "")

# ---- Path ---------------------------------------------------
BASE_DRIVE = "/content/drive/MyDrive/Thesis-Agung_Prawoto-56225110019"
PDF_SOURCE_FOLDER        = f"{BASE_DRIVE}/Dataset"
IMAGE_DESTINATION_FOLDER = f"{BASE_DRIVE}/Dataset/extracted_images"

# Updated CSV input path to use Google Sheets URL
CSV_URL = "<PUBLISHED_SHEET_CSV_URL>"
CSV_INPUT_PATH           = CSV_URL

CSV_OUTPUT_PATH          = f"{BASE_DRIVE}/DPO/dpo_dataset_output.csv"
CSV_CHECKPOINT           = f"{BASE_DRIVE}/DPO/dpo_checkpoint.csv"
STATE_JSON_PATH          = f"{BASE_DRIVE}/DPO/dpo_state.json"      # NEW: status per kode (robust resume, termasuk skip/error)
LOG_PATH                 = f"{BASE_DRIVE}/DPO/dpo_generator.log"

# ---- Model ID -----------------------------------------------
RETRIEVAL_MODEL_ID = "vidore/colpali-v1.3-hf"
COLLECTION_NAME    = "colpali_v13_hf_multivec_rag_index"
LLM_MODEL_ID       = "Qwen/Qwen2-7B-Instruct"
SENTENCE_MODEL_ID  = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

# ---- Hyperparameter (sesuai paper) --------------------------
TOP_K                        = 3      # top-3 context sesuai paper
LOW_RAM_MODE                 = False
MAX_CONTEXT_CHARS_PER_SOURCE = 1200
MAX_NEW_TOKENS               = 350
MARGIN_THRESHOLD             = 0.15   # minimum margin chosen - rejected

# Bobot S_final (grid-search pada 50 held-out query)
W_SEM_GROUNDING = 0.4
W_SEM_RELEVANCE = 0.4
W_LEXICAL       = 0.2

# Temperature kandidat (sesuai paper)
TEMP_CHOSEN_CONSERVATIVE = 0.4
TEMP_CHOSEN_CREATIVE     = 1.2
TEMP_REJECTED            = 1.8

CHECKPOINT_EVERY = 1     # NEW: simpan checkpoint SETIAP baris -- dataset 1500 baris, tiap baris mahal
                          #      (3x generate LLM + retrieval), jadi maks. kerugian saat sesi
                          #      putus/pindah-akun HANYA 1 baris, bukan 10.
MAX_SESSION_HOURS = 5  # NEW: berhenti otomatis & rapi sebelum limit sesi Colab (gratis ~5 jam/12 jam),
                          #      supaya checkpoint pasti tersimpan alih-alih sesi mati paksa (killed).
MAX_RETRY_ERRORS  = 3    # NEW: baris yang gagal (exception) dicoba ulang otomatis di run berikutnya,
                          #      maksimal N kali, lalu ditandai gagal permanen (skip_permanent).

# ---- Logging ------------------------------------------------
os.makedirs(os.path.dirname(LOG_PATH), exist_ok=True)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(LOG_PATH, encoding="utf-8"),
        logging.StreamHandler(),
    ],
)
log = logging.getLogger(__name__)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
log.info(f"Device: {DEVICE}")

# ---- State model --------------------------------------------
retriever_model     = None
retriever_processor = None
llm_model           = None
llm_tokenizer       = None
sentence_model      = None   # SentenceTransformer, selalu di CPU

# ============================================================
# [Cell 2] Utilitas memori (identik dengan kode validasi)
# ============================================================
def clean_mem():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def print_mem_usage(label=""):
    tag = f" [{label}]" if label else ""
    ram = psutil.virtual_memory()
    log.info(
        f"[MEM]{tag} RAM : {ram.used/1024**3:.2f} / "
        f"{ram.total/1024**3:.2f} GB ({ram.percent:.0f}%)"
    )
    if torch.cuda.is_available():
        alloc    = torch.cuda.memory_allocated()  / 1024**3
        reserved = torch.cuda.memory_reserved()   / 1024**3
        total    = torch.cuda.get_device_properties(0).total_memory / 1024**3
        log.info(
            f"[MEM]{tag} VRAM: alloc={alloc:.2f}GB "
            f"reserved={reserved:.2f}GB / total={total:.2f}GB"
        )

## Panduan: Melanjutkan di Akun Google / Sesi Colab Berbeda

Dataset ini 1500 pasangan -- jauh melebihi batas ~5 jam Colab gratis, jadi pipeline
di bawah **wajib** dijalankan bertahap, lanjut-otomatis (resume), bahkan lintas akun.

**Cara kerja resume-nya:** setiap baris yang selesai diproses (baik berhasil, gagal
margin, maupun error) langsung dicatat di dua file pada folder yang sama:

- `MyDrive/.../DPO/dpo_checkpoint.csv` -> data pair DPO yang sudah berhasil.
- `MyDrive/.../DPO/dpo_state.json` -> status SETIAP baris yang pernah dicoba
  (supaya baris yang sudah gagal-margin/sukses TIDAK diproses ulang, dan baris
  yang error DICOBA ULANG otomatis).

Kedua file ini ditulis ulang **setiap 1 baris** (bukan setiap 10), jadi kalau sesi
mati mendadak, paling banyak hanya 1 baris kerja GPU yang hilang.

### Langkah pindah ke akun Google lain

Folder `MyDrive` terikat ke akun yang sedang login di Colab, jadi Drive akun A
**tidak otomatis terlihat** oleh akun B. Pilih salah satu:

**Opsi 1 -- Share folder (paling gampang, tidak perlu upload ulang)**
1. Di akun A, buka Google Drive -> klik kanan folder `DPO` -> **Share** -> tambahkan
   email akun B sebagai **Editor**.
2. Di akun B, buka **Shared with me**, klik kanan folder `DPO` -> **Add shortcut to Drive**
   (pastikan hasilnya ada di path yang SAMA persis: `MyDrive/Thesis-Agung_Prawoto-56225110019/DPO`,
   sesuaikan `BASE_DRIVE` di Cell 1 kalau path shortcut-nya beda).
3. Jalankan notebook di akun B seperti biasa -- `drive.mount()` akan melihat file
   checkpoint yang sama, resume otomatis lanjut dari baris terakhir.

**Opsi 2 -- Download & upload manual**
1. Di akun A: download `dpo_checkpoint.csv` dan `dpo_state.json` dari folder `DPO`.
2. Di akun B: buat folder dengan path Drive yang SAMA PERSIS, lalu upload kedua
   file tersebut ke sana sebelum menjalankan `run_dpo_pipeline()`.
3. Jalankan notebook -- resume otomatis terdeteksi dari kedua file itu.

> Kalau path di akun B berbeda, ubah saja variabel `BASE_DRIVE` di Cell 1 (Imports & Konfigurasi)
> supaya menunjuk ke folder yang berisi kedua file checkpoint tadi -- tidak perlu ubah kode lain.

Jalankan cell **"Cek progres"** di bawah kapan saja untuk melihat berapa baris yang
sudah selesai / tersisa tanpa perlu menjalankan pipeline penuh.

In [ ]:
# ============================================================
# Cek progres (jalankan kapan saja -- TIDAK menjalankan pipeline / GPU)
# ============================================================
def check_progress(
    csv_input:  str = CSV_INPUT_PATH,
    state_path: str = STATE_JSON_PATH,
    checkpoint: str = CSV_CHECKPOINT,
):
    total = len(load_question_csv(csv_input))
    state = load_state(state_path)

    n_success = sum(1 for v in state.values() if v.get("status") == "success")
    n_margin  = sum(1 for v in state.values() if v.get("status") == "skipped_margin")
    n_error   = sum(1 for v in state.values() if v.get("status") == "error")
    n_done    = n_success + n_margin
    n_left    = total - n_done

    print("=" * 55)
    print(f"Total pertanyaan        : {total}")
    print(f"  Sukses (pair DPO)     : {n_success}")
    print(f"  Dilewati (margin)     : {n_margin}")
    print(f"  Error (akan di-retry) : {n_error}")
    print(f"  Sisa belum diproses   : {n_left}")
    print(f"  Estimasi progres      : {100 * n_done / total:.1f}%" if total else "")
    print("=" * 55)
    if os.path.exists(checkpoint):
        print(f"Checkpoint CSV ditemukan -> {checkpoint}")
    else:
        print("Checkpoint CSV belum ada -- pipeline belum pernah dijalankan di lokasi ini.")
    return {"total": total, "success": n_success, "skipped_margin": n_margin,
            "error": n_error, "left": n_left}

# check_progress()   # <- uncomment untuk langsung menjalankan saat cell ini dieksekusi

In [ ]:
# ============================================================
# [Cell 3] Load / Unload ColPali Retriever (identik validasi)
# ============================================================
def load_retriever():
    global retriever_model, retriever_processor
    if retriever_model is not None:
        return
    print_mem_usage("sebelum load retriever")
    log.info(f"[LOAD] Retriever {RETRIEVAL_MODEL_ID} (bfloat16, GPU)...")
    retriever_processor = ColPaliProcessor.from_pretrained(
        RETRIEVAL_MODEL_ID, token=HF_TOKEN
    )
    retriever_model = ColPaliForRetrieval.from_pretrained(
        RETRIEVAL_MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map="cuda:0" if DEVICE == "cuda" else "cpu",
        token=HF_TOKEN,
        low_cpu_mem_usage=True,
    ).eval()
    print_mem_usage("setelah load retriever")

def unload_retriever():
    global retriever_model, retriever_processor
    if retriever_model is None:
        return
    retriever_model, retriever_processor = release_memory(
        retriever_model, retriever_processor
    )
    clean_mem()
    print_mem_usage("setelah unload retriever")

# ============================================================
# [Cell 4] Load / Unload Qwen2-7B-Instruct (identik validasi)
# ============================================================
def load_llm():
    global llm_model, llm_tokenizer
    if llm_model is not None:
        return
    print_mem_usage("sebelum load LLM")
    log.info(f"[LOAD] LLM {LLM_MODEL_ID} (4-bit BnB, GPU)...")
    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    llm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_ID, token=HF_TOKEN)
    llm_model = AutoModelForCausalLM.from_pretrained(
        LLM_MODEL_ID,
        quantization_config=bnb_cfg,
        device_map="auto",
        token=HF_TOKEN,
        low_cpu_mem_usage=True,
    )
    print_mem_usage("setelah load LLM")

def unload_llm():
    global llm_model, llm_tokenizer
    if llm_model is None:
        return
    llm_model, llm_tokenizer = release_memory(llm_model, llm_tokenizer)
    clean_mem()
    print_mem_usage("setelah unload LLM")

# ============================================================
# [Cell 5] Load SentenceTransformer scorer (CPU, sekali saja)
# ============================================================
def load_sentence_model():
    global sentence_model
    if sentence_model is not None:
        return
    log.info(f"[LOAD] SentenceTransformer {SENTENCE_MODEL_ID} (CPU)...")
    sentence_model = SentenceTransformer(SENTENCE_MODEL_ID, device="cpu")
    log.info("[LOAD] SentenceTransformer siap.")

# ============================================================
# [Cell 6] Koneksi Qdrant
# ============================================================
qdrant = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
_info  = qdrant.get_collection(COLLECTION_NAME)
log.info(f"Collection '{COLLECTION_NAME}': {_info.points_count} points")

# ============================================================
# [Cell 7] Embed query untuk retrieval (identik kode validasi)
# ============================================================
def embed_query(query_text: str):
    load_retriever()
    inputs = retriever_processor(
        text=[query_text], return_tensors="pt"
    ).to(retriever_model.device)
    with torch.no_grad():
        output = retriever_model(**inputs)
        result = output.embeddings[0].float().cpu().numpy().tolist()
    del inputs, output
    return result

# ============================================================
# [Cell 8] Parse halaman & retrieve (identik kode validasi)
# ============================================================
PAGE_PATTERN = re.compile(r"_p(\d{4})\.png$", re.IGNORECASE)

def parse_page_number(image_filename: str):
    m = PAGE_PATTERN.search(image_filename or "")
    return int(m.group(1)) if m else None

def retrieve_pages(query: str, top_k: int = TOP_K):
    query_vector = embed_query(query)
    hits = qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        limit=top_k,
        with_payload=True,
    ).points
    del query_vector
    results = []
    for hit in hits:
        payload        = hit.payload or {}
        image_filename = payload.get("file")
        results.append({
            "score":       hit.score,
            "pdf_name":    payload.get("pdf_name"),
            "image_file":  image_filename,
            "image_path":  os.path.join(IMAGE_DESTINATION_FOLDER, image_filename)
                           if image_filename else None,
            "page_number": parse_page_number(image_filename),
        })
    return results

# ============================================================
# [Cell 9] Ekstrak teks PDF + fallback OCR (identik kode validasi)
# ============================================================
def get_page_text_from_pdf(pdf_name: str, page_number: int) -> str:
    if not pdf_name or page_number is None:
        return ""
    pdf_path = os.path.join(PDF_SOURCE_FOLDER, pdf_name + ".pdf")
    if not os.path.exists(pdf_path):
        return ""
    try:
        doc  = fitz.open(pdf_path)
        page = doc[page_number]
        text = page.get_text("text").strip()
        doc.close()
        del doc
        return text
    except Exception as e:
        log.warning(f"Gagal baca {pdf_name}.pdf hal.{page_number}: {e}")
        return ""

def _open_downscaled(image_path: str, max_size: int = 1024) -> Image.Image:
    img = Image.open(image_path).convert("RGB")
    if max(img.size) > max_size:
        img.thumbnail((max_size, max_size), Image.Resampling.LANCZOS)
    return img

def get_page_text_with_ocr_fallback(
    pdf_name: str, page_number: int, image_path: str
) -> str:
    text = get_page_text_from_pdf(pdf_name, page_number)
    if text:
        return text
    if image_path and os.path.exists(image_path):
        try:
            import pytesseract
            img = _open_downscaled(image_path, max_size=2000)
            try:
                return pytesseract.image_to_string(img, lang="ind+eng").strip()
            finally:
                img.close()
                del img
        except ImportError:
            log.warning("pytesseract belum terinstall.")
    return ""

# ============================================================
# [Cell 10] Build context dari halaman yang di-retrieve
# ============================================================
def build_context(retrieved_pages: list) -> str:
    blocks = []
    for i, p in enumerate(retrieved_pages, start=1):
        text = get_page_text_with_ocr_fallback(
            p["pdf_name"], p["page_number"], p["image_path"]
        )
        if not text:
            continue
        text           = text[:MAX_CONTEXT_CHARS_PER_SOURCE]
        halaman_tampil = (p["page_number"] + 1) if p["page_number"] is not None else "?"
        blocks.append(
            f"[Sumber {i}: {p['pdf_name']}.pdf - Halaman {halaman_tampil}"
            f" | skor={p['score']:.3f}]\n{text}"
        )
    return "\n\n".join(blocks)

# ============================================================
# [Cell 11] Stop-words Bahasa Indonesia
#            Curated set sesuai paper: 89 kata (function words umum)
#            Mencakup semua stop-words dari kode validasi
# ============================================================
STOP_WORDS_ID: set = {
    # --- dari kode validasi (sudah dipakai, tidak berubah) ---
    "yang", "dan", "di", "ini", "itu", "dengan", "untuk", "dari", "ke",
    "ada", "tidak", "pada", "adalah", "dalam", "apa", "bagaimana", "berapa",
    "siapa", "kapan", "dimana", "mengapa", "apakah", "juga", "sudah", "akan",
    "bisa", "harus", "boleh", "saya", "anda", "kamu", "kami", "kita", "mereka",
    "dia", "ia", "lebih", "sangat", "saja", "jika", "maka", "atau", "tetapi",
    "namun", "karena", "oleh", "setelah", "sebelum", "ketika", "saat", "hingga",
    "sampai", "melalui", "tentang", "antara", "mengenai", "tersebut", "bahwa",
    "hal", "cara", "serta", "adapun", "seperti", "sebagai", "secara", "telah",
    "sedang", "perlu", "dapat", "sesuai", "atas", "bawah", "tanpa", "setiap",
    "semua", "seluruh",
    # --- tambahan (function words umum Bahasa Indonesia) ----
    "nya", "pun", "lah", "kah", "pula", "agar", "supaya", "bagi",
    "bila", "sehingga", "sejak", "yaitu", "yakni",
}

MIN_TOKEN_LEN = 4   # filter token < 4 karakter (sesuai paper)

def tokenize_for_lexical(text: str) -> set:
    """
    Tokenisasi set-based dengan filter stopword dan panjang minimum.
    Set-based eliminasi duplikasi -- tidak ada token inflation.
    """
    tokens = re.findall(r"[a-zA-Z0-9]+", text.lower())
    return {
        t for t in tokens
        if len(t) >= MIN_TOKEN_LEN and t not in STOP_WORDS_ID
    }

# ============================================================
# [Cell 12] Triple-Dimension Semantic-Lexical Proxy Scoring
#
#   S_final = 0.4 x Sem_grounding + 0.4 x Sem_relevance + 0.2 x Lexical
#
#   Sem_grounding = max(0, cosine(e_response, e_context))
#   Sem_relevance = max(0, cosine(e_response, e_query))
#   Lexical       = |T_response n T_context| / |T_response|
#
#   Semua embedding dihitung dalam 1 forward pass (efisiensi).
# ============================================================
def compute_sfinal(
    response_text: str,
    context_text:  str,
    query_text:    str,
) -> dict:
    """
    Hitung S_final untuk satu kandidat jawaban.
    L2-normalized embedding -> cosine = dot product.
    """
    load_sentence_model()

    # --- 1 forward pass untuk 3 teks sekaligus ---------------
    embeddings = sentence_model.encode(
        [response_text, context_text, query_text],
        normalize_embeddings=True,   # L2-normalized
        convert_to_numpy=True,
        show_progress_bar=False,
    )
    e_response = embeddings[0]
    e_context  = embeddings[1]
    e_query    = embeddings[2]

    # --- Sem_grounding ---------------------------------------
    sem_grounding = float(max(0.0, np.dot(e_response, e_context)))

    # --- Sem_relevance ---------------------------------------
    sem_relevance = float(max(0.0, np.dot(e_response, e_query)))

    # --- Lexical (set-based, anti-inflation) -----------------
    t_response = tokenize_for_lexical(response_text)
    t_context  = tokenize_for_lexical(context_text)
    lexical    = (
        len(t_response & t_context) / len(t_response)
        if t_response else 0.0
    )

    # --- S_final ---------------------------------------------
    s_final = (
        W_SEM_GROUNDING * sem_grounding
        + W_SEM_RELEVANCE * sem_relevance
        + W_LEXICAL       * lexical
    )

    return {
        "sem_grounding": round(sem_grounding, 4),
        "sem_relevance": round(sem_relevance, 4),
        "lexical":       round(lexical,       4),
        "s_final":       round(s_final,       4),
    }

# ============================================================
# [Cell 13] System prompt (sesuai paper verbatim)
# ============================================================
# Chosen: WITH context -- conservative & creative
SYSPROMPT_CHOSEN = (
    "Jawab singkat & padat (min 3 kalimat) sesuai konteks yang diberikan."
)
# Rejected: WITHOUT context -- paksa halusinasi
SYSPROMPT_REJECTED = (
    "Jawab pertanyaan berikut berdasarkan asumsimu sendiri tanpa referensi faktual."
)

# ============================================================
# [Cell 14] Helper generate satu kandidat jawaban
# ============================================================
def _generate_candidate(
    messages:       list,
    temperature:    float,
    max_new_tokens: int = MAX_NEW_TOKENS,
) -> str:
    """Generate satu jawaban dengan temperature tertentu (Qwen2-7B-Instruct)."""
    load_llm()

    # Ambil string dulu (tokenize=False), lalu tokenisasi manual
    # -- perbaikan bug BatchEncoding dari kode validasi, identik
    prompt_text = llm_tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=False
    )
    enc            = llm_tokenizer(prompt_text, return_tensors="pt").to(llm_model.device)
    input_ids      = enc["input_ids"]
    attention_mask = enc.get("attention_mask")

    eos_ids   = [llm_tokenizer.eos_token_id]
    im_end_id = llm_tokenizer.convert_tokens_to_ids("<|im_end|>")
    if (
        im_end_id is not None
        and im_end_id != llm_tokenizer.unk_token_id
        and im_end_id not in eos_ids
    ):
        eos_ids.append(im_end_id)

    with torch.no_grad():
        output_ids = llm_model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=0.9,
            eos_token_id=eos_ids,
        )

    response = llm_tokenizer.decode(
        output_ids[0][input_ids.shape[-1]:], skip_special_tokens=True
    )
    del enc, input_ids, attention_mask, output_ids
    clean_mem()
    return response.strip()

# ============================================================
# [Cell 15] Generate 3 kandidat untuk satu pertanyaan
# ============================================================
def generate_candidates(query: str, context: str) -> list:
    """
    Hasilkan 3 kandidat sesuai paper:
      [0] chosen_conservative : WITH context, T=0.4
      [1] chosen_creative     : WITH context, T=1.2
      [2] rejected            : WITHOUT context, T=1.8
    """
    user_with_ctx = f"Konteks dokumen:\n{context}\n\nPertanyaan: {query}"
    user_no_ctx   = query

    specs = [
        {
            "label":        "chosen_conservative",
            "temperature":  TEMP_CHOSEN_CONSERVATIVE,
            "with_context": True,
            "sysprompt":    SYSPROMPT_CHOSEN,
            "user_content": user_with_ctx,
        },
        {
            "label":        "chosen_creative",
            "temperature":  TEMP_CHOSEN_CREATIVE,
            "with_context": True,
            "sysprompt":    SYSPROMPT_CHOSEN,
            "user_content": user_with_ctx,
        },
        {
            "label":        "rejected",
            "temperature":  TEMP_REJECTED,
            "with_context": False,
            "sysprompt":    SYSPROMPT_REJECTED,
            "user_content": user_no_ctx,
        },
    ]

    candidates = []
    for spec in specs:
        log.info(
            f"  -> Generate [{spec['label']}] "
            f"(T={spec['temperature']}, context={spec['with_context']})..."
        )
        messages = [
            {"role": "system", "content": spec["sysprompt"]},
            {"role": "user",   "content": spec["user_content"]},
        ]
        text = _generate_candidate(messages, temperature=spec["temperature"])
        candidates.append({
            "label":        spec["label"],
            "text":         text,
            "temperature":  spec["temperature"],
            "with_context": spec["with_context"],
        })
        log.info(f"     [{spec['label']}] {len(text)} karakter")

    return candidates

# ============================================================
# [Cell 16] Score semua kandidat & pilih pair DPO
# ============================================================
def select_dpo_pair(candidates: list, context: str, query: str):
    """
    Score setiap kandidat dengan S_final, lalu:
      chosen   = WITH-context skor tertinggi
      rejected = WITHOUT-context
      Return None jika margin < MARGIN_THRESHOLD.
    """
    scored = []
    for c in candidates:
        score_info = compute_sfinal(c["text"], context, query)
        scored.append({**c, **score_info})
        log.info(
            f"  Score [{c['label']}]: "
            f"grounding={score_info['sem_grounding']:.3f}  "
            f"relevance={score_info['sem_relevance']:.3f}  "
            f"lexical={score_info['lexical']:.3f}  "
            f"-> S_final={score_info['s_final']:.3f}"
        )

    with_ctx    = [c for c in scored if     c["with_context"]]
    without_ctx = [c for c in scored if not c["with_context"]]

    if not with_ctx or not without_ctx:
        log.warning("  Kandidat with/without context tidak lengkap -- dilewati.")
        return None

    best_chosen    = max(with_ctx,    key=lambda x: x["s_final"])
    worst_rejected = min(without_ctx, key=lambda x: x["s_final"])

    margin = best_chosen["s_final"] - worst_rejected["s_final"]
    log.info(
        f"  Margin: {best_chosen['s_final']:.3f} - "
        f"{worst_rejected['s_final']:.3f} = {margin:.3f} "
        f"(threshold={MARGIN_THRESHOLD})"
    )

    if margin < MARGIN_THRESHOLD:
        log.warning(
            f"  Margin {margin:.3f} < {MARGIN_THRESHOLD} -> baris DILEWATI."
        )
        return None

    return {
        "chosen":                   best_chosen["text"],
        "score_chosen":             best_chosen["s_final"],
        "temp_chosen":              best_chosen["temperature"],
        "label_chosen":             best_chosen["label"],
        "sem_grounding_chosen":     best_chosen["sem_grounding"],
        "sem_relevance_chosen":     best_chosen["sem_relevance"],
        "lexical_chosen":           best_chosen["lexical"],
        "rejected":                 worst_rejected["text"],
        "score_rejected":           worst_rejected["s_final"],
        "temp_rejected":            worst_rejected["temperature"],
        "sem_grounding_rejected":   worst_rejected["sem_grounding"],
        "sem_relevance_rejected":   worst_rejected["sem_relevance"],
        "lexical_rejected":         worst_rejected["lexical"],
        "score_margin":             round(margin, 4),
    }

# ============================================================
# [Cell 17] Pipeline utama per baris CSV
# ============================================================
def process_one_row(row: dict):
    """
    Proses satu baris CSV menjadi triplet DPO.
    Return None jika margin tidak memenuhi threshold.
    """
    kode       = str(row.get("Kode Pertanyaan", "")).strip()
    topik      = str(row.get("Topik", "")).strip()
    pertanyaan = str(row.get("Pertanyaan", "")).strip()
    sumber     = str(row.get("Sumber File", "")).strip()

    if not pertanyaan:
        raise ValueError(f"[{kode}] Kolom 'Pertanyaan' kosong.")

    # 1) Retrieve
    log.info(f"[{kode}] Retrieve top-{TOP_K} halaman...")
    retrieved = retrieve_pages(pertanyaan, top_k=TOP_K)
    if LOW_RAM_MODE:
        unload_retriever()

    context = build_context(retrieved)
    sources_summary = " | ".join(
        f"{p['pdf_name']}.pdf hal."
        f"{(p['page_number']+1) if p['page_number'] is not None else '?'}"
        f" (skor={p['score']:.3f})"
        for p in retrieved
    )
    log.info(f"[{kode}] Sumber: {sources_summary}")

    # 2) Generate 3 kandidat
    candidates = generate_candidates(pertanyaan, context)
    if LOW_RAM_MODE:
        unload_llm()

    # 3) Score & pilih pair DPO
    pair = select_dpo_pair(candidates, context, pertanyaan)
    if pair is None:
        return None

    return {
        "kode_pertanyaan":          kode,
        "topik":                    topik,
        "pertanyaan":               pertanyaan,
        "sumber_file":              sumber,
        "context_retrieved":        context,
        "sources_summary":          sources_summary,
        "chosen":                   pair["chosen"],
        "score_chosen":             pair["score_chosen"],
        "sem_grounding_chosen":     pair["sem_grounding_chosen"],
        "sem_relevance_chosen":     pair["sem_relevance_chosen"],
        "lexical_chosen":           pair["lexical_chosen"],
        "temp_chosen":              pair["temp_chosen"],
        "label_chosen":             pair["label_chosen"],
        "rejected":                 pair["rejected"],
        "score_rejected":           pair["score_rejected"],
        "sem_grounding_rejected":   pair["sem_grounding_rejected"],
        "sem_relevance_rejected":   pair["sem_relevance_rejected"],
        "lexical_rejected":         pair["lexical_rejected"],
        "temp_rejected":            pair["temp_rejected"],
        "score_margin":             pair["score_margin"],
        "timestamp":                datetime.datetime.now().isoformat(timespec="seconds"),
    }

# ============================================================
# [Cell 18] Skema kolom output DPO
# ============================================================
DPO_FIELDNAMES = [
    "kode_pertanyaan", "topik", "pertanyaan", "sumber_file",
    "context_retrieved", "sources_summary",
    "chosen", "score_chosen",
    "sem_grounding_chosen", "sem_relevance_chosen", "lexical_chosen",
    "temp_chosen", "label_chosen",
    "rejected", "score_rejected",
    "sem_grounding_rejected", "sem_relevance_rejected", "lexical_rejected",
    "temp_rejected", "score_margin", "timestamp",
]

def load_question_csv(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, dtype=str, keep_default_na=False)
    required = {"Kode Pertanyaan", "Topik", "Pertanyaan", "Sumber File"}
    missing  = required - set(df.columns)
    if missing:
        raise ValueError(f"Kolom tidak ditemukan di CSV: {missing}")
    log.info(f"CSV input: {len(df)} baris, kolom: {list(df.columns)}")
    return df

# ============================================================
# [Cell 19] Checkpoint & resume -- VERSI ROBUST (multi-akun / multi-sesi)
# ============================================================
# Dua file disimpan di Drive setiap baris (bukan setiap 10 baris):
#   1. CSV_CHECKPOINT  -> hanya baris yang BERHASIL jadi pair DPO (data asli).
#   2. STATE_JSON_PATH -> status SETIAP kode_pertanyaan yang pernah dicoba
#      (success / skipped_margin / error), termasuk jumlah percobaan.
#      Ini krusial karena CSV_CHECKPOINT saja tidak tahu baris mana yang
#      sudah dicoba tapi gagal margin/error -- tanpa state ini baris itu
#      akan diproses ulang percuma (buang waktu GPU) setiap resume.
#
# Penulisan bersifat ATOMIC (tulis ke file .tmp lalu os.replace) supaya
# checkpoint tidak pernah korup walau sesi mati persis saat menulis.
# ============================================================
def _atomic_write_text(path: str, text: str):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp_path = path + ".tmp"
    with open(tmp_path, "w", encoding="utf-8") as f:
        f.write(text)
    os.replace(tmp_path, path)   # atomic pada filesystem yang sama

def save_checkpoint(records: list, path: str):
    df = pd.DataFrame(records, columns=DPO_FIELDNAMES)
    _atomic_write_text(path, df.to_csv(index=False))
    log.info(f"[CHECKPOINT] CSV disimpan -> {path} ({len(records)} baris)")

def load_state(state_path: str) -> dict:
    """
    state = { kode_pertanyaan: {"status": "success"|"skipped_margin"|"error",
                                 "attempts": int, "timestamp": str} }
    """
    if not os.path.exists(state_path):
        return {}
    try:
        with open(state_path, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception as e:
        log.warning(f"[STATE] Gagal baca {state_path}: {e} -- mulai state kosong.")
        return {}

def save_state(state: dict, state_path: str):
    _atomic_write_text(state_path, json.dumps(state, ensure_ascii=False, indent=2))

def update_state(state: dict, kode: str, status: str, state_path: str):
    prev_attempts = state.get(kode, {}).get("attempts", 0)
    state[kode] = {
        "status":    status,
        "attempts":  prev_attempts + 1,
        "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
    }
    save_state(state, state_path)

def load_already_done(checkpoint_path: str, state_path: str, max_retry_errors: int = 3):
    """
    Resume yang aman untuk pindah akun/sesi.
    Return:
      done_kode        : set kode yang TIDAK perlu diproses lagi
                          (status success / skipped_margin / gagal permanen)
      retry_kode       : set kode berstatus "error" yang AKAN dicoba ulang
      records          : list dict baris pair DPO yang sudah berhasil (dari CSV)
      state            : dict state lengkap (untuk terus diupdate selama run)
    """
    state = load_state(state_path)

    records = []
    if os.path.exists(checkpoint_path):
        try:
            df = pd.read_csv(checkpoint_path, dtype=str, keep_default_na=False)
            records = df.to_dict("records")
        except Exception as e:
            log.warning(f"[RESUME] Gagal baca {checkpoint_path}: {e}")

    done_kode, retry_kode, permanent_fail = set(), set(), set()
    for kode, info in state.items():
        status   = info.get("status")
        attempts = info.get("attempts", 1)
        if status in ("success", "skipped_margin"):
            done_kode.add(kode)
        elif status == "error":
            if attempts >= max_retry_errors:
                permanent_fail.add(kode)
                done_kode.add(kode)
            else:
                retry_kode.add(kode)

    log.info(
        f"[RESUME] {len(done_kode)} baris final (sukses/margin-skip/gagal-permanen), "
        f"{len(retry_kode)} baris error akan dicoba ulang, "
        f"{len(records)} pair DPO sudah tersimpan di CSV."
    )
    if permanent_fail:
        log.warning(f"[RESUME] Gagal permanen (>= {max_retry_errors}x error): {sorted(permanent_fail)}")
    return done_kode, retry_kode, records, state

# ============================================================
# [Cell 20] Pipeline utama -- iterasi seluruh CSV
# ============================================================
def run_dpo_pipeline(
    csv_input:  str   = CSV_INPUT_PATH,
    csv_output: str   = CSV_OUTPUT_PATH,
    checkpoint: str   = CSV_CHECKPOINT,
    state_path: str   = STATE_JSON_PATH,
    max_session_hours: float = MAX_SESSION_HOURS,
    max_retry_errors:  int   = MAX_RETRY_ERRORS,
):
    log.info("=" * 65)
    log.info("Pipeline Generator Dataset DPO -- FASILKOM")
    log.info(
        f"Scoring: S_final = {W_SEM_GROUNDING}xSem_grounding "
        f"+ {W_SEM_RELEVANCE}xSem_relevance + {W_LEXICAL}xLexical"
    )
    log.info(f"Margin threshold   : {MARGIN_THRESHOLD}")
    log.info(f"TOP_K context      : {TOP_K}")
    log.info(
        f"Temperature chosen : conservative={TEMP_CHOSEN_CONSERVATIVE}, "
        f"creative={TEMP_CHOSEN_CREATIVE}"
    )
    log.info(f"Temperature rejected: {TEMP_REJECTED}")
    log.info(f"Checkpoint          : setiap {CHECKPOINT_EVERY} baris -> {checkpoint}")
    log.info(f"State (resume)      : {state_path}")
    log.info(f"Batas sesi          : {max_session_hours} jam (berhenti otomatis & rapi)")
    log.info("=" * 65)

    os.makedirs(os.path.dirname(csv_output), exist_ok=True)

    # Load sentence model sekali di awal (ringan, CPU)
    load_sentence_model()

    df = load_question_csv(csv_input)
    done, retry_kode, records, state = load_already_done(
        checkpoint, state_path, max_retry_errors
    )

    skipped_margin = [k for k, v in state.items() if v.get("status") == "skipped_margin"]
    errors_now     = []
    total          = len(df)
    start_time     = time.monotonic()
    time_budget_s  = max_session_hours * 3600
    stopped_early  = False

    for idx, row in df.iterrows():
        kode = str(row.get("Kode Pertanyaan", "")).strip()

        if kode in done:
            continue   # sukses / margin-skip / gagal-permanen -- tidak diproses lagi

        # ---- Batas waktu sesi: berhenti RAPI sebelum Colab mati paksa ----
        elapsed = time.monotonic() - start_time
        if elapsed > time_budget_s:
            log.warning(
                f"\n[TIME BUDGET] {elapsed/3600:.2f} jam terlampaui "
                f"(batas {max_session_hours} jam). Berhenti secara aman."
            )
            stopped_early = True
            break

        tag = " (RETRY)" if kode in retry_kode else ""
        log.info(f"\n-- [{idx+1}/{total}] Memproses {kode}{tag} "
                  f"(elapsed={elapsed/60:.1f}menit) --")
        print_mem_usage(f"sebelum {kode}")

        try:
            result = process_one_row(row.to_dict())

            if result is None:
                update_state(state, kode, "skipped_margin", state_path)
                done.add(kode)
                skipped_margin.append(kode)
                log.info(f"[{kode}] Dilewati (margin < {MARGIN_THRESHOLD}).")
            else:
                records.append(result)
                update_state(state, kode, "success", state_path)
                done.add(kode)
                log.info(
                    f"[{kode}] OK -- pair DPO disimpan "
                    f"(margin={result['score_margin']:.3f})"
                )

        except Exception as exc:
            log.error(f"[{kode}] ERROR:\n{traceback.format_exc()}")
            update_state(state, kode, "error", state_path)
            errors_now.append({"kode": kode, "error": str(exc)})
            # TIDAK ditambahkan ke `done` selama attempts < max_retry_errors,
            # sehingga akan dicoba ulang otomatis di run/akun berikutnya.

        # Checkpoint CSV -- setiap baris (CHECKPOINT_EVERY=1 secara default)
        if records and len(records) % CHECKPOINT_EVERY == 0:
            save_checkpoint(records, checkpoint)

    # Simpan final (selalu, walau berhenti lebih awal / exception tak terduga)
    save_checkpoint(records, checkpoint)
    final_df = pd.DataFrame(records, columns=DPO_FIELDNAMES)
    final_df.to_csv(csv_output, index=False, encoding="utf-8-sig")

    sisa = total - len(done)

    # ---- Ringkasan ------------------------------------------
    log.info(f"\n{'='*65}")
    log.info("BERHENTI LEBIH AWAL (batas waktu sesi)" if stopped_early else "SELESAI")
    log.info(f"  Total pertanyaan    : {total}")
    log.info(f"  Pair DPO berhasil   : {len(records)}")
    log.info(f"  Dilewati (margin)   : {len(skipped_margin)}")
    log.info(f"  Error (sesi ini)    : {len(errors_now)}")
    log.info(f"  Sisa belum selesai  : {sisa}")
    log.info(f"  Output              : {csv_output}")
    log.info(f"  Checkpoint          : {checkpoint}")
    log.info(f"  State               : {state_path}")
    if skipped_margin:
        log.info(f"  Kode margin-gagal   : {skipped_margin}")
    if errors_now:
        log.warning("  Kode error (akan di-retry otomatis di run berikutnya):")
        for e in errors_now:
            log.warning(f"    {e['kode']}: {e['error']}")
    if sisa > 0:
        log.info(
            "\n  >>> BELUM SELESAI. Untuk lanjut (di akun/sesi yang sama ATAU akun lain):\n"
            f"      1. Pastikan folder Drive '{os.path.dirname(checkpoint)}' berisi\n"
            f"         'dpo_checkpoint.csv' DAN 'dpo_state.json' dari sesi sebelumnya\n"
            "         (copy/upload manual dulu jika pindah akun Google -- lihat catatan\n"
            "         di cell instruksi 'Ganti akun Colab' di atas).\n"
            "      2. Jalankan ulang run_dpo_pipeline() -- baris yang sudah selesai\n"
            "         (sukses/margin-skip) otomatis dilewati, baris error dicoba ulang.\n"
        )
    log.info("=" * 65)

    return final_df

# ============================================================
# [Cell 21] Jalankan pipeline
# ============================================================
if __name__ == "__main__":
    dpo_df = run_dpo_pipeline()

    print("\n-- Preview 3 baris pertama --")
    preview_cols = [
        "kode_pertanyaan", "score_chosen", "score_rejected",
        "score_margin", "label_chosen",
    ]
    print(dpo_df[preview_cols].head(3).to_string(index=False))

# ============================================================
# [Cell 22] (Opsional) Analisis distribusi skor dataset DPO
# ============================================================
def analyze_dpo_scores(csv_path: str = CSV_OUTPUT_PATH):
    """Tampilkan statistik distribusi skor untuk verifikasi kualitas dataset."""
    df = pd.read_csv(csv_path, dtype=str, keep_default_na=False)
    num_cols = [
        "score_chosen", "score_rejected", "score_margin",
        "sem_grounding_chosen", "sem_relevance_chosen", "lexical_chosen",
        "sem_grounding_rejected", "sem_relevance_rejected", "lexical_rejected",
    ]
    for col in num_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    log.info(f"\n-- Analisis Distribusi Skor DPO ({len(df)} pair) --")
    for col in ["score_chosen", "score_rejected", "score_margin"]:
        log.info(
            f"  {col:25s}: mean={df[col].mean():.3f}  "
            f"std={df[col].std():.3f}  "
            f"min={df[col].min():.3f}  "
            f"max={df[col].max():.3f}"
        )
    label_dist = df["label_chosen"].value_counts().to_dict()
    log.info(f"  Distribusi label chosen : {label_dist}")
    log.info(
        f"  Margin >= {MARGIN_THRESHOLD} : "
        f"{(df['score_margin'] >= MARGIN_THRESHOLD).sum()} / {len(df)}"
    )
    log.info("-" * 65)
    return df


# Uncomment untuk menjalankan analisis setelah pipeline:
analyze_dpo_scores()


# ============================================================
# [Cell 23] ABLATION STUDY — Triple-Dimension TPS Scoring
# ============================================================
# Tujuan: Mengisolasi kontribusi individual setiap dimensi TPS
# terhadap kualitas pair selection DPO (scoring-level ablation).
#
# Empat varian scoring diuji pada dataset DPO yang sudah ada
# (86 pair valid) TANPA retraining model -- efisien dan jujur.
#
# Varian:
#   FULL    : 0.4 x SG + 0.4 x SR + 0.2 x Lex  (konfigurasi paper)
#   NO_SG   : 0.0 x SG + 0.5 x SR + 0.5 x Lex  (tanpa Sem_grounding)
#   NO_SR   : 0.5 x SG + 0.0 x SR + 0.5 x Lex  (tanpa Sem_relevance)
#   NO_LEX  : 0.5 x SG + 0.5 x SR + 0.0 x Lex  (tanpa Lexical)
#
# Bobot sisa didistribusikan merata ke 2 dimensi yang tersisa
# agar total tetap 1.0 dan range skor [0,1] tetap valid.
#
# Output:
#   - Tabel perbandingan: jumlah pair valid, avg margin, rank flips
#   - CSV ablation report
#   - Log interpretasi per varian
# ============================================================

ABLATION_VARIANTS = {
    "FULL":   {"w_sg": 0.4, "w_sr": 0.4, "w_lex": 0.2},  # paper config
    "NO_SG":  {"w_sg": 0.0, "w_sr": 0.5, "w_lex": 0.5},  # isolate SG
    "NO_SR":  {"w_sg": 0.5, "w_sr": 0.0, "w_lex": 0.5},  # isolate SR
    "NO_LEX": {"w_sg": 0.5, "w_sr": 0.5, "w_lex": 0.0},  # isolate Lex
}

ABLATION_OUTPUT_PATH = f"{BASE_DRIVE}/DPO/ablation_study_report.csv"
ABLATION_LOG_PATH    = f"{BASE_DRIVE}/DPO/ablation_study.log"


def _rescore_with_weights(
    sg: float, sr: float, lex: float,
    w_sg: float, w_sr: float, w_lex: float,
) -> float:
    """Hitung ulang S_final dari komponen raw dengan bobot varian."""
    return round(w_sg * sg + w_sr * sr + w_lex * lex, 4)


def run_ablation_study(
    dpo_csv_path:    str = CSV_OUTPUT_PATH,
    output_csv_path: str = ABLATION_OUTPUT_PATH,
    margin_threshold: float = MARGIN_THRESHOLD,
) -> pd.DataFrame:
    """
    Jalankan ablation study scoring-level pada dataset DPO yang sudah ada.

    Parameter
    ---------
    dpo_csv_path     : path ke CSV output pipeline utama (dpo_dataset_output.csv)
    output_csv_path  : path untuk menyimpan laporan ablation
    margin_threshold : threshold margin yang sama seperti pipeline utama (0.15)

    Returns
    -------
    DataFrame ringkasan ablation per varian
    """
    # ---- Setup logger ablation ----------------------------------
    ablation_log = logging.getLogger("ablation")
    if not ablation_log.handlers:
        os.makedirs(os.path.dirname(ABLATION_LOG_PATH), exist_ok=True)
        ablation_log.addHandler(
            logging.FileHandler(ABLATION_LOG_PATH, encoding="utf-8")
        )
        ablation_log.addHandler(logging.StreamHandler())
    ablation_log.setLevel(logging.INFO)

    ablation_log.info("=" * 65)
    ablation_log.info("ABLATION STUDY — Triple-Dimension TPS Scoring")
    ablation_log.info(f"Input CSV  : {dpo_csv_path}")
    ablation_log.info(f"Threshold  : margin >= {margin_threshold}")
    ablation_log.info(f"Varian     : {list(ABLATION_VARIANTS.keys())}")
    ablation_log.info("=" * 65)

    # ---- Load dataset DPO ---------------------------------------
    if not os.path.exists(dpo_csv_path):
        ablation_log.error(
            f"File tidak ditemukan: {dpo_csv_path}\n"
            "Jalankan run_dpo_pipeline() terlebih dahulu."
        )
        return pd.DataFrame()

    df = pd.read_csv(dpo_csv_path, dtype=str, keep_default_na=False)
    numeric_cols = [
        "sem_grounding_chosen",  "sem_relevance_chosen",  "lexical_chosen",
        "sem_grounding_rejected","sem_relevance_rejected","lexical_rejected",
        "score_chosen", "score_rejected", "score_margin",
    ]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=numeric_cols)
    total_pairs = len(df)
    ablation_log.info(f"Total pair DPO valid dimuat: {total_pairs}")

    # ---- Rekonstruksi chosen/rejected berdasarkan FULL varian ---
    # Catatan: dataset asli sudah memilih chosen (with-ctx tertinggi)
    # dan rejected (without-ctx). Kita evaluasi ulang MARGIN saja,
    # bukan re-select, karena teks jawaban sudah tetap.
    # Ini adalah scoring-level ablation yang jujur sesuai batasan paper.

    summary_rows    = []
    per_pair_frames = []

    for variant_name, weights in ABLATION_VARIANTS.items():
        w_sg  = weights["w_sg"]
        w_sr  = weights["w_sr"]
        w_lex = weights["w_lex"]

        ablation_log.info(
            f"\n-- Varian: {variant_name} | "
            f"w_SG={w_sg}, w_SR={w_sr}, w_Lex={w_lex} --"
        )

        # Hitung skor ulang untuk chosen dan rejected
        df[f"score_chosen_{variant_name}"] = df.apply(
            lambda r: _rescore_with_weights(
                r["sem_grounding_chosen"], r["sem_relevance_chosen"],
                r["lexical_chosen"], w_sg, w_sr, w_lex
            ), axis=1
        )
        df[f"score_rejected_{variant_name}"] = df.apply(
            lambda r: _rescore_with_weights(
                r["sem_grounding_rejected"], r["sem_relevance_rejected"],
                r["lexical_rejected"], w_sg, w_sr, w_lex
            ), axis=1
        )
        df[f"margin_{variant_name}"] = (
            df[f"score_chosen_{variant_name}"]
            - df[f"score_rejected_{variant_name}"]
        ).round(4)

        # Pair yang masih valid (margin >= threshold)
        valid_mask   = df[f"margin_{variant_name}"] >= margin_threshold
        valid_count  = valid_mask.sum()
        invalid_count = total_pairs - valid_count

        # Rata-rata margin
        avg_margin     = df[f"margin_{variant_name}"].mean()
        avg_margin_valid = df.loc[valid_mask, f"margin_{variant_name}"].mean() \
                           if valid_count > 0 else 0.0

        # Rank flip: pair dimana chosen varian < rejected varian (margin < 0)
        rank_flips = (df[f"margin_{variant_name}"] < 0).sum()

        # Delta vs FULL (hanya tersedia setelah varian FULL diproses)
        if variant_name == "FULL":
            delta_valid  = 0
            delta_margin = 0.0
        else:
            full_valid   = next(
                r["valid_pairs"] for r in summary_rows if r["variant"] == "FULL"
            )
            full_margin  = next(
                r["avg_margin"] for r in summary_rows if r["variant"] == "FULL"
            )
            delta_valid  = valid_count - full_valid
            delta_margin = round(avg_margin - full_margin, 4)

        ablation_log.info(
            f"  Pair valid (margin>={margin_threshold}) : "
            f"{valid_count}/{total_pairs} ({100*valid_count/total_pairs:.1f}%)"
        )
        ablation_log.info(f"  Pair gugur (margin<{margin_threshold})   : {invalid_count}")
        ablation_log.info(f"  Rank flip (margin<0)                    : {rank_flips}")
        ablation_log.info(f"  Avg margin (semua pair)                 : {avg_margin:.4f}")
        ablation_log.info(f"  Avg margin (pair valid saja)            : {avg_margin_valid:.4f}")
        if variant_name != "FULL":
            ablation_log.info(
                f"  Delta pair valid vs FULL                : {delta_valid:+d}"
            )
            ablation_log.info(
                f"  Delta avg margin vs FULL                : {delta_margin:+.4f}"
            )

        summary_rows.append({
            "variant":           variant_name,
            "w_sg":              w_sg,
            "w_sr":              w_sr,
            "w_lex":             w_lex,
            "valid_pairs":       int(valid_count),
            "invalid_pairs":     int(invalid_count),
            "rank_flips":        int(rank_flips),
            "avg_margin":        round(avg_margin, 4),
            "avg_margin_valid":  round(avg_margin_valid, 4),
            "pct_valid":         round(100 * valid_count / total_pairs, 2),
            "delta_valid_vs_full":  delta_valid if variant_name != "FULL" else "-",
            "delta_margin_vs_full": delta_margin if variant_name != "FULL" else "-",
        })

        # Simpan per-pair detail
        pair_detail = df[[
            "kode_pertanyaan",
            f"score_chosen_{variant_name}",
            f"score_rejected_{variant_name}",
            f"margin_{variant_name}",
        ]].copy()
        pair_detail["variant"]      = variant_name
        pair_detail["pair_valid"]   = valid_mask
        pair_detail["rank_flip"]    = df[f"margin_{variant_name}"] < 0
        pair_detail = pair_detail.rename(columns={
            f"score_chosen_{variant_name}":  "score_chosen",
            f"score_rejected_{variant_name}":"score_rejected",
            f"margin_{variant_name}":        "margin",
        })
        per_pair_frames.append(pair_detail)

    # ---- Ringkasan ablation -------------------------------------
    summary_df = pd.DataFrame(summary_rows)

    ablation_log.info("\n" + "=" * 65)
    ablation_log.info("RINGKASAN ABLATION STUDY")
    ablation_log.info("=" * 65)
    ablation_log.info(
        summary_df[[
            "variant","valid_pairs","pct_valid","rank_flips",
            "avg_margin","delta_valid_vs_full","delta_margin_vs_full"
        ]].to_string(index=False)
    )

    # ---- Interpretasi otomatis ----------------------------------
    ablation_log.info("\n-- INTERPRETASI OTOMATIS --")
    full_valid = summary_df.loc[
        summary_df["variant"] == "FULL", "valid_pairs"
    ].values[0]

    for _, row in summary_df.iterrows():
        if row["variant"] == "FULL":
            continue
        removed_dim = {
            "NO_SG":  "Semantic Grounding (Dim2)",
            "NO_SR":  "Semantic Relevance (Dim1)",
            "NO_LEX": "Lexical Discipline (Dim3)",
        }[row["variant"]]
        loss = full_valid - row["valid_pairs"]
        pct_loss = 100 * loss / full_valid if full_valid > 0 else 0
        flip_note = (
            f", {row['rank_flips']} rank flip"
            if row["rank_flips"] > 0 else ""
        )
        ablation_log.info(
            f"  Tanpa {removed_dim}: "
            f"-{loss} pair valid ({pct_loss:.1f}% hilang){flip_note} "
            f"| delta margin={row['delta_margin_vs_full']:+.4f}"
        )

    # Dimensi paling kritis (paling banyak pair hilang saat dihapus)
    no_variants = summary_df[summary_df["variant"] != "FULL"].copy()
    most_critical = no_variants.loc[
        no_variants["valid_pairs"].idxmin(), "variant"
    ]
    dim_label = {
        "NO_SG":  "Semantic Grounding",
        "NO_SR":  "Semantic Relevance",
        "NO_LEX": "Lexical Discipline",
    }[most_critical]
    ablation_log.info(
        f"\n  >>> DIMENSI PALING KRITIS: {dim_label} "
        f"(penghapusannya menyebabkan pair valid paling sedikit)"
    )
    ablation_log.info("=" * 65)

    # ---- Simpan output ------------------------------------------
    os.makedirs(os.path.dirname(output_csv_path), exist_ok=True)

    # Sheet 1: ringkasan per varian
    summary_df.to_csv(output_csv_path, index=False, encoding="utf-8-sig")
    ablation_log.info(f"Ringkasan ablation disimpan -> {output_csv_path}")

    # Sheet 2: detail per pair (semua varian digabung)
    per_pair_df = pd.concat(per_pair_frames, ignore_index=True)
    detail_path = output_csv_path.replace(".csv", "_per_pair_detail.csv")
    per_pair_df.to_csv(detail_path, index=False, encoding="utf-8-sig")
    ablation_log.info(f"Detail per-pair disimpan    -> {detail_path}")

    return summary_df


def print_ablation_table(summary_df: pd.DataFrame):
    """
    Cetak tabel ablation yang siap disalin ke paper (format teks).
    Sesuai Table format JUITA.
    """
    if summary_df.empty:
        print("Ablation summary kosong. Jalankan run_ablation_study() dulu.")
        return

    print("\n" + "=" * 75)
    print("TABLE: Ablation Study — Kontribusi Setiap Dimensi TPS")
    print("=" * 75)
    header = (
        f"{'Variant':<10} {'w_SG':>5} {'w_SR':>5} {'w_Lex':>6} "
        f"{'Valid Pairs':>12} {'% Valid':>8} {'Avg Margin':>11} "
        f"{'Rank Flips':>11} {'Δ Pairs':>8} {'Δ Margin':>9}"
    )
    print(header)
    print("-" * 75)
    for _, row in summary_df.iterrows():
        delta_pairs  = str(row["delta_valid_vs_full"])  if row["variant"] != "FULL" else "-"
        delta_margin = str(row["delta_margin_vs_full"]) if row["variant"] != "FULL" else "-"
        print(
            f"{row['variant']:<10} {row['w_sg']:>5.1f} {row['w_sr']:>5.1f} "
            f"{row['w_lex']:>6.1f} {row['valid_pairs']:>12} "
            f"{row['pct_valid']:>7.1f}% {row['avg_margin']:>11.4f} "
            f"{row['rank_flips']:>11} {delta_pairs:>8} {delta_margin:>9}"
        )
    print("=" * 75)
    print(
        "Catatan: 'Δ' = perubahan vs varian FULL (konfigurasi paper).\n"
        "Rank Flip = pair dimana chosen score < rejected score (margin < 0).\n"
        "Varian NO_XX = dimensi XX dihapus, bobot didistribusikan merata ke 2 dimensi lain."
    )


# ============================================================
# [Cell 24] Jalankan Ablation Study
# ============================================================
# Cara pakai:
#   1. Pastikan run_dpo_pipeline() sudah selesai dan CSV output tersedia.
#   2. Jalankan cell ini.
#   3. Hasil tersimpan di CSV dan log, serta dicetak di bawah.
#
# CATATAN: Ini adalah scoring-level ablation (tanpa retraining).
# Setiap varian mengevaluasi ulang margin dari komponen skor
# yang sudah tersimpan di dataset -- tidak memerlukan GPU/LLM.
# ============================================================

if __name__ == "__main__":
    # Jalankan ablation setelah pipeline utama selesai
    print("\n" + "="*65)
    print("Menjalankan Ablation Study TPS...")
    print("="*65)

    ablation_summary = run_ablation_study(
        dpo_csv_path=CSV_OUTPUT_PATH,
        output_csv_path=ABLATION_OUTPUT_PATH,
        margin_threshold=MARGIN_THRESHOLD,
    )

    print_ablation_table(ablation_summary)

# Atau jalankan secara mandiri (tanpa pipeline):
# ablation_summary = run_ablation_study()
# print_ablation_table(ablation_summary)

In [ ]:
# ============================================================
# [Cell 23] ABLATION STUDY — Triple-Dimension TPS Scoring
# ============================================================
# Tujuan: Mengisolasi kontribusi individual setiap dimensi TPS
# terhadap kualitas pair selection DPO (scoring-level ablation).
#
# Empat varian scoring diuji pada dataset DPO yang sudah ada
# (86 pair valid) TANPA retraining model -- efisien dan jujur.
#
# Varian:
#   FULL    : 0.4 x SG + 0.4 x SR + 0.2 x Lex  (konfigurasi paper)
#   NO_SG   : 0.0 x SG + 0.5 x SR + 0.5 x Lex  (tanpa Sem_grounding)
#   NO_SR   : 0.5 x SG + 0.0 x SR + 0.5 x Lex  (tanpa Sem_relevance)
#   NO_LEX  : 0.5 x SG + 0.5 x SR + 0.0 x Lex  (tanpa Lexical)
#
# Bobot sisa didistribusikan merata ke 2 dimensi yang tersisa
# agar total tetap 1.0 dan range skor [0,1] tetap valid.
#
# Output:
#   - Tabel perbandingan: jumlah pair valid, avg margin, rank flips
#   - CSV ablation report
#   - Log interpretasi per varian
# ============================================================

ABLATION_VARIANTS = {
    "FULL":   {"w_sg": 0.4, "w_sr": 0.4, "w_lex": 0.2},  # paper config
    "NO_SG":  {"w_sg": 0.0, "w_sr": 0.5, "w_lex": 0.5},  # isolate SG
    "NO_SR":  {"w_sg": 0.5, "w_sr": 0.0, "w_lex": 0.5},  # isolate SR
    "NO_LEX": {"w_sg": 0.5, "w_sr": 0.5, "w_lex": 0.0},  # isolate Lex
}

ABLATION_OUTPUT_PATH = f"{BASE_DRIVE}/DPO/ablation_study_report.csv"
ABLATION_LOG_PATH    = f"{BASE_DRIVE}/DPO/ablation_study.log"


def _rescore_with_weights(
    sg: float, sr: float, lex: float,
    w_sg: float, w_sr: float, w_lex: float,
) -> float:
    """Hitung ulang S_final dari komponen raw dengan bobot varian."""
    return round(w_sg * sg + w_sr * sr + w_lex * lex, 4)


def run_ablation_study(
    dpo_csv_path:    str = CSV_OUTPUT_PATH,
    output_csv_path: str = ABLATION_OUTPUT_PATH,
    margin_threshold: float = MARGIN_THRESHOLD,
) -> pd.DataFrame:
    """
    Jalankan ablation study scoring-level pada dataset DPO yang sudah ada.

    Parameter
    ---------
    dpo_csv_path     : path ke CSV output pipeline utama (dpo_dataset_output.csv)
    output_csv_path  : path untuk menyimpan laporan ablation
    margin_threshold : threshold margin yang sama seperti pipeline utama (0.15)

    Returns
    -------
    DataFrame ringkasan ablation per varian
    """
    # ---- Setup logger ablation ----------------------------------
    ablation_log = logging.getLogger("ablation")
    if not ablation_log.handlers:
        os.makedirs(os.path.dirname(ABLATION_LOG_PATH), exist_ok=True)
        ablation_log.addHandler(
            logging.FileHandler(ABLATION_LOG_PATH, encoding="utf-8")
        )
        ablation_log.addHandler(logging.StreamHandler())
    ablation_log.setLevel(logging.INFO)

    ablation_log.info("=" * 65)
    ablation_log.info("ABLATION STUDY — Triple-Dimension TPS Scoring")
    ablation_log.info(f"Input CSV  : {dpo_csv_path}")
    ablation_log.info(f"Threshold  : margin >= {margin_threshold}")
    ablation_log.info(f"Varian     : {list(ABLATION_VARIANTS.keys())}")
    ablation_log.info("=" * 65)

    # ---- Load dataset DPO ---------------------------------------
    if not os.path.exists(dpo_csv_path):
        ablation_log.error(
            f"File tidak ditemukan: {dpo_csv_path}\n"
            "Jalankan run_dpo_pipeline() terlebih dahulu."
        )
        return pd.DataFrame()

    df = pd.read_csv(dpo_csv_path, dtype=str, keep_default_na=False)
    numeric_cols = [
        "sem_grounding_chosen",  "sem_relevance_chosen",  "lexical_chosen",
        "sem_grounding_rejected","sem_relevance_rejected","lexical_rejected",
        "score_chosen", "score_rejected", "score_margin",
    ]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=numeric_cols)
    total_pairs = len(df)
    ablation_log.info(f"Total pair DPO valid dimuat: {total_pairs}")

    # ---- Rekonstruksi chosen/rejected berdasarkan FULL varian ---
    # Catatan: dataset asli sudah memilih chosen (with-ctx tertinggi)
    # dan rejected (without-ctx). Kita evaluasi ulang MARGIN saja,
    # bukan re-select, karena teks jawaban sudah tetap.
    # Ini adalah scoring-level ablation yang jujur sesuai batasan paper.

    summary_rows    = []
    per_pair_frames = []

    for variant_name, weights in ABLATION_VARIANTS.items():
        w_sg  = weights["w_sg"]
        w_sr  = weights["w_sr"]
        w_lex = weights["w_lex"]

        ablation_log.info(
            f"\n-- Varian: {variant_name} | "
            f"w_SG={w_sg}, w_SR={w_sr}, w_Lex={w_lex} --"
        )

        # Hitung skor ulang untuk chosen dan rejected
        df[f"score_chosen_{variant_name}"] = df.apply(
            lambda r: _rescore_with_weights(
                r["sem_grounding_chosen"], r["sem_relevance_chosen"],
                r["lexical_chosen"], w_sg, w_sr, w_lex
            ), axis=1
        )
        df[f"score_rejected_{variant_name}"] = df.apply(
            lambda r: _rescore_with_weights(
                r["sem_grounding_rejected"], r["sem_relevance_rejected"],
                r["lexical_rejected"], w_sg, w_sr, w_lex
            ), axis=1
        )
        df[f"margin_{variant_name}"] = (
            df[f"score_chosen_{variant_name}"]
            - df[f"score_rejected_{variant_name}"]
        ).round(4)

        # Pair yang masih valid (margin >= threshold)
        valid_mask   = df[f"margin_{variant_name}"] >= margin_threshold
        valid_count  = valid_mask.sum()
        invalid_count = total_pairs - valid_count

        # Rata-rata margin
        avg_margin     = df[f"margin_{variant_name}"].mean()
        avg_margin_valid = df.loc[valid_mask, f"margin_{variant_name}"].mean() \
                           if valid_count > 0 else 0.0

        # Rank flip: pair dimana chosen varian < rejected varian (margin < 0)
        rank_flips = (df[f"margin_{variant_name}"] < 0).sum()

        # Delta vs FULL (hanya tersedia setelah varian FULL diproses)
        if variant_name == "FULL":
            delta_valid  = 0
            delta_margin = 0.0
        else:
            full_valid   = next(
                r["valid_pairs"] for r in summary_rows if r["variant"] == "FULL"
            )
            full_margin  = next(
                r["avg_margin"] for r in summary_rows if r["variant"] == "FULL"
            )
            delta_valid  = valid_count - full_valid
            delta_margin = round(avg_margin - full_margin, 4)

        ablation_log.info(
            f"  Pair valid (margin>={margin_threshold}) : "
            f"{valid_count}/{total_pairs} ({100*valid_count/total_pairs:.1f}%)"
        )
        ablation_log.info(f"  Pair gugur (margin<{margin_threshold})   : {invalid_count}")
        ablation_log.info(f"  Rank flip (margin<0)                    : {rank_flips}")
        ablation_log.info(f"  Avg margin (semua pair)                 : {avg_margin:.4f}")
        ablation_log.info(f"  Avg margin (pair valid saja)            : {avg_margin_valid:.4f}")
        if variant_name != "FULL":
            ablation_log.info(
                f"  Delta pair valid vs FULL                : {delta_valid:+d}"
            )
            ablation_log.info(
                f"  Delta avg margin vs FULL                : {delta_margin:+.4f}"
            )

        summary_rows.append({
            "variant":           variant_name,
            "w_sg":              w_sg,
            "w_sr":              w_sr,
            "w_lex":             w_lex,
            "valid_pairs":       int(valid_count),
            "invalid_pairs":     int(invalid_count),
            "rank_flips":        int(rank_flips),
            "avg_margin":        round(avg_margin, 4),
            "avg_margin_valid":  round(avg_margin_valid, 4),
            "pct_valid":         round(100 * valid_count / total_pairs, 2),
            "delta_valid_vs_full":  delta_valid if variant_name != "FULL" else "-",
            "delta_margin_vs_full": delta_margin if variant_name != "FULL" else "-",
        })

        # Simpan per-pair detail
        pair_detail = df[[
            "kode_pertanyaan",
            f"score_chosen_{variant_name}",
            f"score_rejected_{variant_name}",
            f"margin_{variant_name}",
        ]].copy()
        pair_detail["variant"]      = variant_name
        pair_detail["pair_valid"]   = valid_mask
        pair_detail["rank_flip"]    = df[f"margin_{variant_name}"] < 0
        pair_detail = pair_detail.rename(columns={
            f"score_chosen_{variant_name}":  "score_chosen",
            f"score_rejected_{variant_name}":"score_rejected",
            f"margin_{variant_name}":        "margin",
        })
        per_pair_frames.append(pair_detail)

    # ---- Ringkasan ablation -------------------------------------
    summary_df = pd.DataFrame(summary_rows)

    ablation_log.info("\n" + "=" * 65)
    ablation_log.info("RINGKASAN ABLATION STUDY")
    ablation_log.info("=" * 65)
    ablation_log.info(
        summary_df[[
            "variant","valid_pairs","pct_valid","rank_flips",
            "avg_margin","delta_valid_vs_full","delta_margin_vs_full"
        ]].to_string(index=False)
    )

    # ---- Interpretasi otomatis ----------------------------------
    ablation_log.info("\n-- INTERPRETASI OTOMATIS --")
    full_valid = summary_df.loc[
        summary_df["variant"] == "FULL", "valid_pairs"
    ].values[0]

    for _, row in summary_df.iterrows():
        if row["variant"] == "FULL":
            continue
        removed_dim = {
            "NO_SG":  "Semantic Grounding (Dim2)",
            "NO_SR":  "Semantic Relevance (Dim1)",
            "NO_LEX": "Lexical Discipline (Dim3)",
        }[row["variant"]]
        loss = full_valid - row["valid_pairs"]
        pct_loss = 100 * loss / full_valid if full_valid > 0 else 0
        flip_note = (
            f", {row['rank_flips']} rank flip"
            if row["rank_flips"] > 0 else ""
        )
        ablation_log.info(
            f"  Tanpa {removed_dim}: "
            f"-{loss} pair valid ({pct_loss:.1f}% hilang){flip_note} "
            f"| delta margin={row['delta_margin_vs_full']:+.4f}"
        )

    # Dimensi paling kritis (paling banyak pair hilang saat dihapus)
    no_variants = summary_df[summary_df["variant"] != "FULL"].copy()
    most_critical = no_variants.loc[
        no_variants["valid_pairs"].idxmin(), "variant"
    ]
    dim_label = {
        "NO_SG":  "Semantic Grounding",
        "NO_SR":  "Semantic Relevance",
        "NO_LEX": "Lexical Discipline",
    }[most_critical]
    ablation_log.info(
        f"\n  >>> DIMENSI PALING KRITIS: {dim_label} "
        f"(penghapusannya menyebabkan pair valid paling sedikit)"
    )
    ablation_log.info("=" * 65)

    # ---- Simpan output ------------------------------------------
    os.makedirs(os.path.dirname(output_csv_path), exist_ok=True)

    # Sheet 1: ringkasan per varian
    summary_df.to_csv(output_csv_path, index=False, encoding="utf-8-sig")
    ablation_log.info(f"Ringkasan ablation disimpan -> {output_csv_path}")

    # Sheet 2: detail per pair (semua varian digabung)
    per_pair_df = pd.concat(per_pair_frames, ignore_index=True)
    detail_path = output_csv_path.replace(".csv", "_per_pair_detail.csv")
    per_pair_df.to_csv(detail_path, index=False, encoding="utf-8-sig")
    ablation_log.info(f"Detail per-pair disimpan    -> {detail_path}")

    return summary_df


def print_ablation_table(summary_df: pd.DataFrame):
    """
    Cetak tabel ablation yang siap disalin ke paper (format teks).
    Sesuai Table format JUITA.
    """
    if summary_df.empty:
        print("Ablation summary kosong. Jalankan run_ablation_study() dulu.")
        return

    print("\n" + "=" * 75)
    print("TABLE: Ablation Study — Kontribusi Setiap Dimensi TPS")
    print("=" * 75)
    header = (
        f"{'Variant':<10} {'w_SG':>5} {'w_SR':>5} {'w_Lex':>6} "
        f"{'Valid Pairs':>12} {'% Valid':>8} {'Avg Margin':>11} "
        f"{'Rank Flips':>11} {'Δ Pairs':>8} {'Δ Margin':>9}"
    )
    print(header)
    print("-" * 75)
    for _, row in summary_df.iterrows():
        delta_pairs  = str(row["delta_valid_vs_full"])  if row["variant"] != "FULL" else "-"
        delta_margin = str(row["delta_margin_vs_full"]) if row["variant"] != "FULL" else "-"
        print(
            f"{row['variant']:<10} {row['w_sg']:>5.1f} {row['w_sr']:>5.1f} "
            f"{row['w_lex']:>6.1f} {row['valid_pairs']:>12} "
            f"{row['pct_valid']:>7.1f}% {row['avg_margin']:>11.4f} "
            f"{row['rank_flips']:>11} {delta_pairs:>8} {delta_margin:>9}"
        )
    print("=" * 75)
    print(
        "Catatan: 'Δ' = perubahan vs varian FULL (konfigurasi paper).\n"
        "Rank Flip = pair dimana chosen score < rejected score (margin < 0).\n"
        "Varian NO_XX = dimensi XX dihapus, bobot didistribusikan merata ke 2 dimensi lain."
    )


# ============================================================
# [Cell 24] Jalankan Ablation Study
# ============================================================
# Cara pakai:
#   1. Pastikan run_dpo_pipeline() sudah selesai dan CSV output tersedia.
#   2. Jalankan cell ini.
#   3. Hasil tersimpan di CSV dan log, serta dicetak di bawah.
#
# CATATAN: Ini adalah scoring-level ablation (tanpa retraining).
# Setiap varian mengevaluasi ulang margin dari komponen skor
# yang sudah tersimpan di dataset -- tidak memerlukan GPU/LLM.
# ============================================================

if __name__ == "__main__":
    # Jalankan ablation setelah pipeline utama selesai
    print("\n" + "="*65)
    print("Menjalankan Ablation Study TPS...")
    print("="*65)

    ablation_summary = run_ablation_study(
        dpo_csv_path=CSV_OUTPUT_PATH,
        output_csv_path=ABLATION_OUTPUT_PATH,
        margin_threshold=MARGIN_THRESHOLD,
    )

    print_ablation_table(ablation_summary)

# Atau jalankan secara mandiri (tanpa pipeline):
ablation_summary = run_ablation_study()
print_ablation_table(ablation_summary)